**Helper for triples manual checking**

In [2]:
#  --- owl:sameAs ---

# src/07_link_audit.ipynb (single cell) — save CSV report of external-link triples
from pathlib import Path
import csv
from rdflib import Graph, Namespace

# --- Paths (notebook is in /src, TTL is in /output) ---
KG_PATH = Path("..") / "output" / "merged_kg_a2.ttl"
OUT_DIR = Path("..") / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "links_report.csv"

# --- Namespaces ---
OWL  = Namespace("http://www.w3.org/2002/07/owl#")
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")
RDFS = Namespace("http://www.w3.org/2000/01/rdf-schema#")

# --- Load KG ---
g = Graph()
g.parse(str(KG_PATH), format="turtle")

# --- Predicates to audit ---
preds = [OWL.sameAs, SKOS.exactMatch, SKOS.closeMatch, SKOS.relatedMatch, RDFS.seeAlso]

# --- Collect rows ---
rows = []
for p in preds:
    for s, _, o in g.triples((None, p, None)):
        s_label = next(g.objects(s, RDFS.label), "")
        o_label = next(g.objects(o, RDFS.label), "")
        rows.append([str(p), str(s), str(o), str(s_label), str(o_label)])

# --- Print counts ---
def count(pred_uri: str) -> int:
    return sum(1 for r in rows if r[0] == pred_uri)

print("Loaded:", KG_PATH)
print("Triples:", len(g))
print("owl:sameAs:", count(str(OWL.sameAs)))
print("skos:exactMatch:", count(str(SKOS.exactMatch)))
print("skos:closeMatch:", count(str(SKOS.closeMatch)))
print("skos:relatedMatch:", count(str(SKOS.relatedMatch)))
print("rdfs:seeAlso:", count(str(RDFS.seeAlso)))
print("Total link triples (these predicates):", len(rows))

# --- Write CSV report ---
with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["predicate", "subject", "object", "subject_label", "object_label"])
    w.writerows(rows)

print("Wrote:", OUT_CSV)

Loaded: ..\output\merged_kg_a2.ttl
Triples: 3236
owl:sameAs: 0
skos:exactMatch: 17
skos:closeMatch: 0
skos:relatedMatch: 0
rdfs:seeAlso: 0
Total link triples (these predicates): 17
Wrote: ..\output\links_report.csv
